# Finance Tracker — NB-BERT Fine-tuning
**Norwegian transaction categorizer** — 7 enterprise B2B categories

## Before running:
1. **Runtime → Change runtime type → T4 GPU**
2. Upload your 3 Parquet files to Google Drive under `My Drive/finance_ml/data/`:
   - `train.parquet`
   - `val.parquet`
   - `test.parquet`
3. Run all cells top to bottom

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
# This MUST show a GPU. If it says 'CPU only', go to:
# Runtime → Change runtime type → Hardware accelerator → T4 GPU
import torch

if torch.cuda.is_available():
    print(f"✓ GPU ready: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU found! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
# transformers: HuggingFace library — downloads NB-BERT, handles training loop
# pyarrow:      reads Parquet files (our training data format)
# accelerate:   required by HuggingFace Trainer for GPU utilization
!pip install -q transformers pyarrow accelerate scikit-learn
print("✓ Dependencies installed")

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────────
# This opens a browser popup asking for permission to access your Drive.
# Your Parquet files must be at: My Drive/finance_ml/data/
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DATA_DIR   = Path('/content/drive/MyDrive/finance_ml/data')
OUTPUT_DIR = Path('/content/drive/MyDrive/finance_ml/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify all 3 splits exist
for split in ['train', 'val', 'test']:
    f = DATA_DIR / f'{split}.parquet'
    assert f.exists(), f"Missing: {f}\nUpload your Parquet files to My Drive/finance_ml/data/"
    print(f"✓ Found {f.name}")

In [ ]:
# ── Cell 4: Constants ─────────────────────────────────────────────────────────
# These mirror src/ml/data/label_transactions.py exactly.
# We inline them here so the notebook has zero local dependencies.

CATEGORIES = [
    "saas_software",         # SaaS platforms — Salesforce, Slack, SAP, GitHub
    "cloud_infrastructure",  # Cloud compute/storage — AWS, GCP, Azure, Cloudflare
    "consulting_services",   # Professional services — Big4, boutique firms, IT consulting
    "travel_expenses",       # Business travel — flights, hotels, rideshare
    "office_facilities",     # Office operations — supplies, co-working, facilities
    "marketing_advertising", # Paid marketing — Google Ads, Meta, agencies
    "telecommunications",    # Corporate comms — mobile plans, internet, VoIP
]

CATEGORY_TO_ID = {cat: idx for idx, cat in enumerate(CATEGORIES)}
ID_TO_CATEGORY = {idx: cat for cat, idx in CATEGORY_TO_ID.items()}
NUM_LABELS     = len(CATEGORIES)

# NbAiLab/nb-bert-base:
# - Trained by Norwegian AI Lab on Norwegian text (newspapers, books, web)
# - 12 transformer layers, 768 hidden dimensions, 178M parameters
# - Already understands Norwegian company names as entities
# - Fine-tuning adds a classification head: 768 → 7 categories
MODEL_NAME = "NbAiLab/nb-bert-base"
MAX_LENGTH = 64  # Norwegian transactions are short — even 20 tokens is generous

print(f"✓ {NUM_LABELS} categories: {CATEGORIES}")

In [ ]:
# ── Cell 5: Dataset class ─────────────────────────────────────────────────────
# PyTorch needs data wrapped in a Dataset class.
# __len__() tells the DataLoader how many examples exist.
# __getitem__(i) returns one tokenized example as a dict of tensors.
#
# Tokenization converts text → integer IDs:
#   'SALESFORCE CRM INV-123456' → [101, 8921, 3456, ..., 102, 0, 0]
#   attention_mask marks real tokens (1) vs padding (0)

import pyarrow.parquet as pq
import torch
from torch.utils.data import Dataset

class TransactionDataset(Dataset):
    def __init__(self, parquet_path: Path, tokenizer):
        table = pq.read_table(parquet_path)
        df = table.to_pandas()

        self._labels = df['label_id'].tolist()

        # Tokenize all at once (batch = faster than one-by-one)
        self._encodings = tokenizer(
            df['description'].tolist(),
            truncation=True,
            padding='max_length',
            max_length=MAX_LENGTH,
            return_tensors='pt',
        )
        print(f"  Loaded {len(self._labels)} examples from {parquet_path.name}")

    def __len__(self):
        return len(self._labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self._encodings['input_ids'][idx],
            'attention_mask': self._encodings['attention_mask'][idx],
            'labels':         torch.tensor(self._labels[idx], dtype=torch.long),
        }

print("✓ Dataset class defined")

In [ ]:
# ── Cell 6: Metrics function ──────────────────────────────────────────────────
# Called by Trainer after each epoch to compute accuracy + weighted F1.
# Weighted F1 = harmonic mean of precision/recall, weighted by class frequency.
# It's harder to game than accuracy — a model that ignores minority classes gets penalized.

import numpy as np
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy':    accuracy_score(labels, predictions),
        'f1_weighted': f1_score(labels, predictions, average='weighted'),
    }

print("✓ Metrics function defined")

In [ ]:
# ── Cell 7: Load model + tokenizer ───────────────────────────────────────────
# This downloads NB-BERT from HuggingFace Hub (~440MB). Takes ~1-2 min.
# Cached in Colab session — won't re-download if you re-run this cell.
#
# AutoModelForSequenceClassification adds a classification head on top of BERT:
#   BERT encoder (178M params, frozen knowledge of Norwegian)
#     → [CLS] token hidden state (768 dimensions)
#     → Dropout(0.1)
#     → Linear(768 → 7)   ← this layer learns during fine-tuning
#     → softmax → category probabilities

from transformers import AutoModelForSequenceClassification, AutoTokenizer

print(f"Downloading {MODEL_NAME} from HuggingFace (~440MB)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID_TO_CATEGORY,
    label2id=CATEGORY_TO_ID,
)
total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Model loaded — {total_params:,} parameters")
print(f"✓ Tokenizer vocab size: {tokenizer.vocab_size:,}")

In [ ]:
# ── Cell 8: Load datasets ─────────────────────────────────────────────────────
print("Loading datasets...")
train_dataset = TransactionDataset(DATA_DIR / 'train.parquet', tokenizer)
val_dataset   = TransactionDataset(DATA_DIR / 'val.parquet',   tokenizer)
test_dataset  = TransactionDataset(DATA_DIR / 'test.parquet',  tokenizer)
print(f"✓ Total: {len(train_dataset)} train / {len(val_dataset)} val / {len(test_dataset)} test")

In [ ]:
# ── Cell 9: Training ──────────────────────────────────────────────────────────
# TrainingArguments = all hyperparameters in one place.
#
# Key settings explained:
#   learning_rate=2e-5  → BERT paper's recommended value for fine-tuning
#   warmup_ratio=0.1    → first 10% of steps: lr rises 0 → 2e-5 (prevents early instability)
#   fp16=True           → 16-bit floats: 2x faster, 2x less VRAM, ~0.01% accuracy loss
#   load_best_model_at_end → after training, restore the checkpoint with best val F1
#   early_stopping_patience=2 → stop if val F1 doesn't improve for 2 epochs

from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
import json

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_weighted',
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    dataloader_num_workers=2,
    report_to='none',
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting fine-tuning... (~1-2 hours on T4)")
print("Watch the loss drop in the output below.")
print("─" * 60)
trainer.train()

In [ ]:
# ── Cell 10: Evaluate on test set ─────────────────────────────────────────────
# The test set is the 'sealed envelope' — never seen during training or validation.
# This is the honest measure of how well the model generalizes.
print("Final evaluation on test set:")
test_results = trainer.evaluate(test_dataset)
print(f"  F1 (weighted): {test_results.get('eval_f1_weighted', 'N/A'):.4f}")
print(f"  Accuracy:      {test_results.get('eval_accuracy', 'N/A'):.4f}")

In [ ]:
# ── Cell 11: Save model to Drive ──────────────────────────────────────────────
# Saves to Google Drive so it persists after the Colab session ends.
# Download from Drive → place at: src/ml/models/final_model/
# Then run: python -m src.ml.train.export_onnx

final_model_dir = OUTPUT_DIR / 'final_model'
trainer.save_model(str(final_model_dir))
tokenizer.save_pretrained(str(final_model_dir))

# Save test results alongside the model
with open(final_model_dir / 'test_results.json', 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'categories': CATEGORIES,
        'test_f1_weighted': test_results.get('eval_f1_weighted'),
        'test_accuracy':    test_results.get('eval_accuracy'),
        'num_train':        len(train_dataset),
        'num_val':          len(val_dataset),
        'num_test':         len(test_dataset),
    }, f, indent=2)

print(f"✓ Model saved to Google Drive at: finance_ml/outputs/final_model/")
print()
print("Next steps:")
print("  1. Download the 'final_model' folder from Google Drive")
print("  2. Place it at: src/ml/models/final_model/")
print("  3. Run locally: python -m src.ml.train.export_onnx")